# 하이브리드 검색과 Reranker로 Retriever 성능 높이기
- 지금까지는 문서를 로드하고, 청크로 나누고, 임베딩한 뒤 Chroma retriever 로 검색했습니다.
- 이번에는 **BM25 키워드 검색**, **Chroma 벡터 검색**, **EnsembleRetriever 하이브리드 검색**, **Reranker** 를 붙여 검색 품질을 높입니다.
- 핵심 목표는 "1차로 넓게 후보를 찾고, 2차로 질문에 더 맞는 후보를 위로 올리는 것"입니다.


## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [ ]:
# 필요한 라이브러리 설치
# uv add -qU rank-bm25 langchain-classic langchain-cohere kiwipiepy langchain-chroma langchain-openai python-dotenv

## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
OPENAI_API_KEY=sk-...
COHERE_API_KEY=...
```

- `OPENAI_API_KEY`: 임베딩, LLM-as-reranker, RAG 답변 생성에 사용
- `COHERE_API_KEY`: Cohere Rerank 실습에 사용. 없으면 해당 셀은 건너뜁니다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")

## 2. 회사 문서 로드 + 청크 분할

- 이전 RAG 실습과 같은 `data/company_docs` 폴더 사용
- 여러 문서를 한 번에 로드한 뒤, 검색에 적합한 크기로 청킹

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = Path("data/company_docs")

In [ ]:
def load_text_documents(data_dir: Path):
    all_docs = []
    for pattern in ["**/*.txt", "**/*.md"]:
        loader = DirectoryLoader(
            str(data_dir),
            glob=pattern,
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"},
        )
        all_docs.extend(loader.load())
    return all_docs

raw_docs = load_text_documents(DATA_DIR)

for doc in raw_docs:
    source = doc.metadata.get("source", "")
    doc.metadata["category"] = Path(source).stem

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ".", " ", ""],
)
split_docs = text_splitter.split_documents(raw_docs)

for i, doc in enumerate(split_docs):
    doc.metadata["doc_id"] = f"company-doc-{i:04d}"

In [ ]:
print(f"원본 문서 수: {len(raw_docs)}")
print(f"검색용 청크 수: {len(split_docs)}")
print("첫 번째 청크 metadata:", split_docs[0].metadata)
print(split_docs[0].page_content[:300])

## 3. Chroma 벡터 검색 준비

- 벡터 검색은 질문과 문서의 **의미 유사도** 를 잘 잡음
- 대신 고유명사, 코드, 정확한 키워드 매칭에는 약할 수 있음

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

collection_name = "company_docs_hybrid_rerank"

# 반복 실행 대비: 같은 collection이 있으면 지우고 다시 생성
reset_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
)
reset_store.delete_collection()

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    ids=[doc.metadata["doc_id"] for doc in split_docs],
    collection_name=collection_name,
)

In [ ]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("저장 문서 수:", vectorstore._collection.count())

## 4. BM25 키워드 검색 준비

- 한국어는 공백만으로 단어를 나누면 조사와 어미 때문에 BM25 품질이 흔들릴 수 있기 때문에, 가능하면 `kiwipiepy` 같은 형태소 분석기를 붙임


| 검색 방식 | 잘하는 것 | 약한 것 |
|---|---|---|
| **BM25** | 고유명사, 코드, 정확한 단어 매칭 | 동의어, 표현 변형, 의미 유사도 |
| **벡터 검색** | 동의어, 문맥, 자연어 의미 | 최신 고유명사, 약어, 정확한 문자열 |



In [ ]:
#uv add rank_bm25 kiwipiepy

In [ ]:
from langchain_community.retrievers import BM25Retriever


def korean_tokenizer(text: str) -> list[str]:
    """명사, 동사, 형용사, 외국어, 숫자 중심으로 BM25 토큰을 만듭니다."""
    meaningful_tags = {"NNG", "NNP", "VV", "VA", "SL", "SN"} # 품사: 일반명사(NNG), 고유명사(NNP), 동사(VV), 형용사(VA), 알파벳(SL) 
    return [
        token.form.lower()
        for token in kiwi.tokenize(text)
        if token.tag in meaningful_tags
    ]


sample = " "
print("예시 토큰:", )

## 5. BM25 검색과 벡터 검색 비교

- BM25는 정확한 단어가 있는 문서를 잘 찾고, 벡터 검색은 표현이 조금 달라도 의미가 가까운 문서를 잘 찾음

In [ ]:
from langchain_core.documents import Document


def print_docs(title: str, docs: list[Document], max_chars: int = 220):
    print(f"\n=== {title} ({len(docs)}개) ===")
    for i, doc in enumerate(docs, start=1):
        category = doc.metadata.get("category", "unknown")
        text = doc.page_content.replace("\n", " ")
        print(f"\n[{i}] category={category}")
        print(text[:max_chars])


question = " "

bm25_results = 
vector_results = 

print(f"질문: {question}")
print_docs("BM25 결과", )
print_docs("벡터 결과", )

## 6. 하이브리드 검색, BM25 + 벡터 동시에

- `EnsembleRetriever` 는 여러 retriever 의 결과를 RRF(Reciprocal Rank Fusion) 방식으로 합침
- RRF는 각 retriever 의 원점수를 직접 섞지 않고, **순위**를 기준으로 합침
- 즉, RRF 는 각 retriever 가 매긴 순위 (1등, 2등, ...) 를 받아서 "여러 retriever 에서 모두 상위에 든 문서" 가 가장 높은 점수를 받게 함.
- 그래서 BM25 점수와 벡터 유사도 점수처럼 스케일이 다른 검색 결과도 비교적 안정적으로 결합할 수 있음

| 방식 | 강점 | 약점 |
|------|------|------|
| **BM25 (키워드)** | 정확한 고유명사 매칭 | 의미 유사성 약함 |
| **Chroma (의미)** | 문맥 이해 | 정확한 키워드 매칭 약함 |
| **Hybrid** | 두 방식의 장점 결합 | 가중치 튜닝 필요 |

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = 



In [ ]:
for q in [
    "재택근무 신청 방법",           
    "법인카드 영수증 제출 기한",           
    "API 키는 어디에 보관해야 해?",  
]:
    print(f"\n질문: {q}")
    print(f"한국어 토큰: {korean_tokenizer(q)}")
    print_docs("Hybrid 결과", hybrid_retriever.invoke(q), max_chars=180)

## 7. 가중치 실험

- 하이브리드 검색은 도메인에 따라 BM25와 벡터의 비율 조정

| 도메인 | 출발 가중치 |
|---|---|
| 기술 문서, 코드, 고유명사 많음 | `[0.6, 0.4]` 또는 `[0.7, 0.3]` |
| 자연어 Q&A, 표현 다양함 | `[0.3, 0.7]` 또는 `[0.4, 0.6]` |
| 처음 시작할 때 | `[0.5, 0.5]` |

In [ ]:
weight_sets = {
    "BM25 only": [1.0, 0.0],
    "BM25 우선": [0.7, 0.3],
    "균형": [0.5, 0.5],
    "Vector 우선": [0.3, 0.7],
    "Vector only": [0.0, 1.0],
}

question = " "

for label, weights in weight_sets.items():
    retriever = 


    docs =
    top = 
    print(f"\n[{label}] weights={weights}")
    print(f"top category={top.metadata.get('category')}")
    print(top.page_content.replace("\n", " ")[:220])

## 8. Reranker, 후보를 한 번 더 정렬

- 하이브리드 검색은 1차 후보를 넓게 찾는 역할
- Reranker는 질문과 후보 문서를 **쌍으로 직접 비교**해서 더 관련 있는 문서를 위로 올림

- 운영에서 자주 쓰는 패턴:

```text
1차 retriever: BM25 + Vector 로 후보 20~30개 확보
2차 reranker: 후보를 3~5개로 재정렬/압축
최종 LLM: 재정렬된 문서만 참고해 답변
```

In [ ]:
# uv add langchain_cohere

In [ ]:
load_dotenv()

print("COHERE_API_KEY:", "있음" if os.getenv("COHERE_API_KEY") else "없음")

In [ ]:
# reranker가 만들어지지 않는 경우를 대비해 기본값을 None으로 둠
rerank_retriever = None

# Cohere에서 제공하는 reranker 모델



compressor = 


rerank_retriever =




question = ""

print(f"질문: {question}")

print_docs("Hybrid 상위 후보", , max_chars=160)

print_docs("Cohere Rerank 결과", , max_chars=220)


## 9. LLM-as-reranker 대안

- Cohere 같은 전용 reranker가 없을 때는 LLM에게 "질문과 문서의 관련도를 0~10점으로 평가"하게 만들 수 있음
- 비용·속도는 떨어지지만 외부 키 없이 됨

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = 

SCORE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", " "),
    ("user", "질문: {question}\n\n문서: {doc}"),
])


scorer = 


def llm_rerank(question: str, candidates: list[Document], top_n: int = 3):
    scored = []
    for d in candidates:
        try:
            score = float(scorer.invoke({"question": question, "doc": d.page_content}).strip())
        except ValueError:
            score = 0.0
        scored.append((d, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]


In [ ]:
question = 
candidates = 


print_docs("Hybrid 후보", candidates, max_chars=150)

reranked = llm_rerank(question, candidates, top_n=3)
if reranked:
    print("\n=== LLM Rerank 결과 ===")
    for i, (doc, score) in enumerate(reranked, start=1):
        print(f"\n[{i}] score={score:.1f}, category={doc.metadata.get('category')}")
        print(doc.page_content.replace("\n", " ")[:220])

## 10. Query 변형으로 검색 질의 개선
- 사용자 질문은 항상 검색에 좋은 형태가 아님
- 예를 들어 "그거 어디서 승인받아?" 같은 질문은 문서의 실제 표현과 거리가 멂
- Query 변형은 LLM으로 질문을 더 검색 친화적인 표현으로 바꾼 뒤 retriever 에 넣는 방식
- 여기서는 간단한 **Multi-Query** 방식을 직접 구현
    - 한 질문을 여러 검색 질의로 확장하고, 결과를 중복 제거해 합침

### 10.1. Multi-Query Retriever
- LLM 이 원 질문을 N 개 다른 표현으로 바꿔서 각각 검색 → 결과 합집합

In [ ]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-5.4-mini")

mq_retriever = 



import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

results = 

print(f"\n총 {len(results)} 개 문서:")
for d in results:
    print(f"  {d.page_content[:70]}")

### 10.2. HyDE (Hypothetical Document Embeddings)
- "질문" 보다 "답 문서" 가 vectorstore 안 문서와 더 비슷할 거라는 직관
- LLM 이 가상 답을 먼저 생성 → 그걸로 검색

In [ ]:
HYDE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", " "),
    ("user", " "),
])

hyde_chain = 


def hyde_search(question: str, k: int = 3):
    hypothetical = hyde_chain.invoke({"question": question})
    print(f"가상 답: {hypothetical[:80]}")
    # 가상 답을 쿼리로 사용
    return retriever.invoke(hypothetical)


question = 
print(f"질문: {question}")

results = 
print("\n=== HyDE 검색 결과 ===")
for d in results:
    print(f"  {d.page_content[:70]}")

## 11. 개선된 retriever 를 RAG 체인에 연결

- Reranker가 준비되어 있으면 `rerank_retriever` 를 사용하고, 없으면 `hybrid_retriever` 를 사용
- 검색기만 바꿔도 RAG 체인의 나머지 구조는 그대로 유지할 수 있음

In [ ]:
# LCEL 체인에서 사용자의 입력 질문을 그대로 다음 단계로 넘기기 위한 Runnable
from langchain_core.runnables import RunnablePassthrough

# 검색된 Document 리스트를 LLM 프롬프트에 넣기 좋은 문자열로 변환하는 함수
def format_docs(docs: list[Document]) -> str:
    if not docs:
        return "검색된 참고 자료가 없습니다."

    formatted = []
    for i, doc in enumerate(docs, start=1):
        formatted.append(
            f"[{i}] category={doc.metadata.get('category')}, source={doc.metadata.get('source')}\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(formatted)

## 12. 정리

- BM25는 정확한 키워드, 고유명사, 코드, 약어 검색에 강합니다.
- 벡터 검색은 표현이 달라도 의미가 가까운 문서를 찾는 데 강합니다.
- 하이브리드 검색은 BM25와 벡터 검색을 RRF 기반으로 합쳐 1차 후보 품질을 높입니다.
- Reranker는 후보 문서를 질문과 직접 비교해 2차로 재정렬합니다.
- Query 변형은 사용자의 모호한 질문을 검색 친화적인 여러 질의로 확장합니다.
- 실무 기본 패턴은 `Query 변형 → Hybrid 후보 검색 → Rerank → RAG 답변` 입니다.

## [실습]

1. `bm25_retriever.k`, `vector_retriever`의 `k` 값을 3, 5, 10으로 바꾸고 결과를 비교합니다.
2. `weights=[0.7, 0.3]`, `[0.5, 0.5]`, `[0.3, 0.7]` 로 같은 질문 5개를 검색해 어떤 질문에서 차이가 큰지 정리합니다.
3. `korean_tokenizer`에서 태그 집합을 바꿔 BM25 결과가 어떻게 달라지는지 확인합니다.
4. Cohere Rerank의 `top_n`을 1, 3, 5로 바꿔 RAG 답변 차이를 비교합니다.
5. Query 변형 + BM25 + 하이브리드 + Reranker 4단계를 한 chain 으로 묶고 RAG 답변을 확인합니다.